In [1]:
import string
import requests
import re
import numpy as np

In [2]:
def get_data():
    resp = requests.get("https://www.gutenberg.org/files/35/35-0.txt")
    return resp.text

txt = get_data()

In [4]:
def clean_text(text):
    strings2replace = [
                 '\r\n\r\nâ\x80\x9c', # new paragraph
                 'â\x80\x9c',         # open quote
                 'â\x80\x9d',         # close quote
                 '\r\n',              # new line
                 'â\x80\x94',         # hyphen
                 'â\x80\x99',         # single apostrophe
                 'â\x80\x98',         # single quote
                 '_',                 # underscore, used for stressing
                 ]

    for rpl in strings2replace:
        rexp = re.compile(r'%s'%rpl)
        text = rexp.sub(' ', text)


    # remove non-ASCII characters
    text = re.sub(r'[^\x00-\x7F]+', ' ', text)

    # remove numbers
    text = re.sub(r'\d+','',text)

    # and make everything lower-case
    text = text.lower()

    return text

In [54]:
text = clean_text(txt)
actual_text = text[515:]

actual_text

'the time traveller (for so it will be convenient to speak of him) was expounding a recondite matter to us. his pale grey eyes shone and twinkled, and his usually pale face was flushed and animated. the fire burnt brightly, and the soft radiance of the incandescent lights in the lilies of silver caught the bubbles that flashed and passed in our glasses. our chairs, being his patents, embraced and caressed us rather than submitted to be sat upon, and there was that luxurious after-dinner atmosphere, when thought runs gracefully free of the trammels of precision. and he put it to us in this way marking the points with a lean forefinger as we sat and lazily admired his earnestness over this new paradox (as we thought it) and his fecundity.   you must follow me carefully. i shall have to controvert one or two ideas that are almost universally accepted. the geometry, for instance, they taught you at school is founded on a misconception.    is not that rather a large thing to expect us to be

In [55]:
all_chars = list(set(actual_text))
all_chars.sort()
vocab = {c:i for i,c in enumerate(all_chars)}
print(len(vocab))
chars_in_txt = list(actual_text)
print(f"Vocab len = {len(vocab)}")
print(f"Text len = {len(actual_text)}")

39
Vocab len = 39
Text len = 179258


In [56]:
def get_most_freq_pair(chars_in_txt):
    pairs = {}
    max_pair = ''
    max_freq = 1
    for c in range(len(chars_in_txt)-1):
        pair = chars_in_txt[c] + chars_in_txt[c+1]
        freq = pairs.get(pair, 0) + 1
        if freq > max_freq:
            max_freq = freq
            max_pair = pair
        pairs[pair] = freq
    return (max_pair, max_freq)


In [57]:
def get_max_vocab_token_idx(vocab_dict: dict):
    return max(vocab_dict.values())
    
def update_vocab(vocab_dic, token, token_idx):
    vocab_dic[token] = token_idx

In [58]:
def update_text_with_new_pair(chars_in_txt, pair_2_update):
    new_txt_chars = []
    i=0
    while (i < len(chars_in_txt)-1):
        pair = chars_in_txt[i] + chars_in_txt[i+1]
        if pair == pair_2_update:
            new_txt_chars.append(pair)
            i += 2
        else:    
            new_txt_chars.append(chars_in_txt[i])
            i += 1
    if i < len(chars_in_txt):
        new_txt_chars.append(chars_in_txt[i])
  
    return new_txt_chars

In [72]:
# print(f"Initial Vocab size = {len(vocab)}")
print(f"Initial txt length size = {len(chars_in_txt)}")

curr_max_v_tok_idx = get_max_vocab_token_idx(vocab)
tmp_txt = chars_in_txt
itr = 20000
while(itr > 0):
    pair, freq = get_most_freq_pair(tmp_txt)
    # print(f"Max pair = {pair} freq = {freq}")
    curr_max_v_tok_idx+=1
    update_vocab(vocab, pair, curr_max_v_tok_idx)
    tmp_txt = update_text_with_new_pair(tmp_txt, pair)
    itr -= 1
print(f"Max pair = {pair} freq = {freq}")
print(f"length of txt after replacement {len(tmp_txt)}")
print(f"Vocab size = {len(vocab)}")
print(tmp_txt)
# print("".join(tmp_txt))
# print(vocab)

Initial txt length size = 179258
Max pair =  freq = 1
length of txt after replacement 30512
Vocab size = 7455
['the time traveller ', '(', 'for', ' so ', 'it ', 'will be ', 'convenient', ' to speak', ' of ', 'him', ')', ' was expound', 'ing a ', 're', 'cond', 'ite ', 'matter', ' to ', 'us', '. his ', 'pale ', 'grey ', 'eyes shone ', 'and', ' twink', 'l', 'ed, and ', 'his ', 'usually ', 'pale ', 'face was ', 'flu', 'sh', 'ed and', ' animat', 'ed. the ', 'fire ', 'burnt ', 'brightly', ', and the s', 'oft ', 'ra', 'di', 'ance ', 'of the ', 'in', 'cand', 'es', 'cent ', 'light', 's in the ', 'li', 'lies ', 'of s', 'ilver ', 'caught the ', 'bu', 'b', 'bles', ' that ', 'flash', 'ed and pass', 'ed in ', 'our ', 'glass', 'es. ', 'our ', 'chair', 's, ', 'being ', 'his ', 'p', 'at', 'ent', 's, ', 'em', 'br', 'ac', 'ed and ', 'caress', 'ed us ', 'rather than', ' sub', 'mitt', 'ed to be ', 'sat upon', ', and', ' there was', ' that ', 'lu', 'x', 'ur', 'ious', ' after', '-', 'dinner', ' at', 'mosp', 

In [77]:
def encoder(words, w2i):
    return [w2i[word.lower()] for word in words]

def decoder(idxs, i2w):
    return " ".join([i2w[idx] for idx in idxs])


In [5]:
from simple_bpe_tokenizer import SimpleBPETokenizer

tokzr = SimpleBPETokenizer(txt)
tokzr.train(1000)

Initial txt length size = 179773
Max pair =  a min freq = 16
length of txt after replacement 56442
Vocab size = 1039
